In [18]:
import numpy as np
import iris
from iris import cube
import iris.coord_categorisation
import iris.analysis.cartography
import glob
import warnings
from iris.util import equalise_attributes 
from iris.util import unify_time_units
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec as gspec
from matplotlib.lines import Line2D

In [19]:
### Becky CMIP6 functions

#
def combine_netCDF_model(directory, model):

    list_files = glob.glob(directory)
    list_files = np.array(list_files)
    newlist = np.sort(list_files)

    Cubelist = iris.cube.CubeList([])

    for i in range(0, len(newlist)):

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            warnings.simplefilter('ignore', UserWarning)

            cube = iris.load_cube(newlist[i])
            Cubelist.append(cube)

    unify_time_units(Cubelist)
    equalise_attributes(Cubelist)

    new_cube = Cubelist.concatenate_cube()

    return new_cube

#
def combine_netCDF_cmip6(directory, model):

    list_files = glob.glob(directory)
    list_files = np.array(list_files)
    newlist = np.sort(list_files)

    Cubelist = iris.cube.CubeList([])

    for i in range(0, len(newlist)):

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            warnings.simplefilter('ignore', UserWarning)
            
            cube = iris.load_cube(newlist[i])

            cube.coord('latitude').attributes = {}
            cube.coord('longitude').attributes = {}  
            if i == 0:
                metadata1 = cube.metadata
            else:
                cube.metadata = metadata1
            
            if model=='IPSL-CM6A-LR' or model=='CNRM-ESM2-1':
                 cube.coord('latitude').guess_bounds()
                 cube.coord('longitude').guess_bounds()
         
            # CESM2 bound issue fix
            if (model=='CESM2') & (i==0):
                lat_data = cube.coord('latitude').points
                lon_data = cube.coord('longitude').points
                lat_bounds = cube.coord('latitude').bounds
                lon_bounds = cube.coord('longitude').bounds
            elif (model=='CESM2') & (i>0):
                cube.coord('latitude').points = lat_data
                cube.coord('longitude').points = lon_data
                cube.coord('latitude').bounds = lat_bounds
                cube.coord('longitude').bounds = lon_bounds
    
            if model=='IPSL-CM6A-LR':
                cube.coord('time').attributes.pop('time_origin')
            
            Cubelist.append(cube)

    unify_time_units(Cubelist)
    equalise_attributes(Cubelist)

    for cube in Cubelist:
        lon_bounds = Cubelist[0].coord('longitude').bounds
        cube.coord('longitude').bounds = lon_bounds

    for i, cube in enumerate(Cubelist):
        if cube.coord('time').units == Cubelist[0].coord('time').units:
            pass
        else:
            print(i)
            
    new_cube = Cubelist.concatenate_cube()

    return new_cube

#
def combine_netCDF_co2(directory):

    list_files = glob.glob(directory)
    list_files = np.array(list_files)
    newlist = np.sort(list_files)

    Cubelist = iris.cube.CubeList([])

    for i in range(0, len(newlist)):

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            warnings.simplefilter('ignore', UserWarning)
            
            cube = iris.load_cube(newlist[i])

            if i == 0:
                metadata1 = cube.metadata
            else:
                cube.metadata = metadata1
            
            Cubelist.append(cube)

    unify_time_units(Cubelist)
    equalise_attributes(Cubelist)

    for i, cube in enumerate(Cubelist):
        if cube.coord('time').units == Cubelist[0].coord('time').units:
            pass
        else:
            print(i)
            
    new_cube = Cubelist.concatenate_cube()

    return new_cube


#
def open_netCDF(new_cube):

    iris.coord_categorisation.add_year(new_cube, 'time', name='year') # add year
    iris.coord_categorisation.add_month(new_cube, 'time', name ='month') # add month
    iris.coord_categorisation.add_month(new_cube, 'time', name ='decade') # add month

    return new_cube

#
def annual_average(new_cube):

    annual_average_cube = new_cube.aggregated_by('year', iris.analysis.MEAN)

    return annual_average_cube

#
def global_total_percentage(cubein, landfrac=None, latlon_cons=None):

    cube = cubein.copy()
    if landfrac is not None:
        try:
            cube.data = cube.data * (landfrac.data/100)
        except:
            landfrac = landfrac.extract(latlon_cons)
            cube.data = cube.data * (landfrac.data/100)

    if cube.coord('latitude').bounds is not None:
        pass
    else:
        cube.coord('latitude').guess_bounds()
        cube.coord('longitude').guess_bounds()

    weights = iris.analysis.cartography.area_weights(cube)

    cube_areaweight = cube.collapsed(['latitude', 'longitude'], iris.analysis.SUM, weights=weights)/1e12

    return cube_areaweight

#
def area_average(cube, region):

    lon1, lon2, lat1, lat2 = region[0], region[1], region[2], region[3] 
    cube = cube.intersection(longitude=(lon1, lon2),latitude=(lat1, lat2))

    weights = iris.analysis.cartography.area_weights(cube)
    cube = cube.collapsed(['latitude','longitude'], iris.analysis.MEAN, weights=weights)

    return cube


# def centered_moving_average(data, average_length):
#     if average_length < 1:
#         raise ValueError("average_length must be >= 1")
#     if average_length > len(data):
#         raise ValueError("average_length larger than data length")
    
#     kernel = np.ones(average_length) / average_length
#     rolling_avg = np.convolve(data, kernel, mode='valid')

#     return rolling_avg
def centered_moving_average(data, window=20):
    if window % 2 != 0:
        raise ValueError("This function expects an even window size.")
        
    half_window = window // 2

    # Pad the array on both sides
    padded = np.pad(data, (half_window, half_window), mode='edge')
    
    # Compute moving average using convolution
    kernel = np.ones(window) / window
    smoothed = np.convolve(padded, kernel, mode='valid')
    
    return smoothed


def clean_and_convert(x):
    if isinstance(x, (bytes, np.bytes_)):
        x = x.decode()
    x = x.strip("'")  # remove surrounding single quotes
    try:
        return float(x)
    except ValueError:
        return np.nan

In [20]:
### Historical & hist-bgc, carbon stocks

# CMIP6 models
cmip6_models = ['ACCESS-ESM1-5', 'CanESM5', 'CNRM-ESM2-1', 'MIROC-ES2L', 'NorESM2-LM', 'UKESM1-0-LL']
n_models = len(cmip6_models)

histbgc_beta = np.zeros((len(cmip6_models)))
histbgc_gamma = np.zeros((len(cmip6_models)))

# CMIP6 models loop
for model_i in range(n_models):
        model = cmip6_models[model_i]
        print(model)

        landfraction = combine_netCDF_model('/Volumes/Extreme SSD/DATA/cmip6_data/sftlf_fx_'+model+'_historical*', model)

        # hist-bgc cLand
        cLand_bgc_cube = combine_netCDF_cmip6('/Volumes/Extreme SSD/DATA/CMIP_histbgc/cLand_Emon_'+model+'_hist-bgc*', model)
        cLand_bgc_cube = open_netCDF(cLand_bgc_cube)
        cLand_bgc_cube = annual_average(cLand_bgc_cube)
        cLand_bgc_cube = global_total_percentage(cLand_bgc_cube, landfrac=landfraction, latlon_cons=None)
        cLand_bgc_data = cLand_bgc_cube.data

        # Time dimension
        if model == 'ACCESS-ESM1-5':
                time_dimension = cLand_bgc_cube.coord('year').points
                time_dimension = time_dimension - 100

        # historical cLand
        cLand_cou_cube = combine_netCDF_cmip6('/Volumes/Extreme SSD/DATA/CMIP_histbgc/cLand_Emon_'+model+'_historical*', model)
        cLand_cou_cube = open_netCDF(cLand_cou_cube)
        cLand_cou_cube = annual_average(cLand_cou_cube)
        cLand_cou_cube = global_total_percentage(cLand_cou_cube, landfrac=landfraction, latlon_cons=None)
        cLand_cou_data = cLand_cou_cube.data

        # Temperature (tas)
        tas_cou_cube = combine_netCDF_cmip6('/Volumes/Extreme SSD/DATA/CMIP_histbgc/tas_Amon_'+model+'_historical*', model)
        tas_cou_cube = open_netCDF(tas_cou_cube)
        tas_cou_cube = annual_average(tas_cou_cube)
        tas_cou_cube = area_average(tas_cou_cube - 273.15, [0, 360, -90,  90])
        tas_cou_data = tas_cou_cube.data
        tas_cou_data = centered_moving_average(tas_cou_data, 20)

        # co2 array
        co2_cube = combine_netCDF_co2('/Volumes/Extreme SSD/DATA/CMIP_histbgc/mole-fraction-of-carbon-dioxide-in-air_input4MIPs_GHGConcentrations_CMIP_UoM-CMIP-1-2-0_gr1-GMNHSH_0000-2014.nc')
        co2_data = co2_cube.data
        co2_array = co2_data[:,0]
        #print(co2_array, co2_array.shape)

        # beta
        dC_bgc = cLand_bgc_data[160] - cLand_bgc_data[0]
        dCo2 = co2_array[160] - co2_array[0]
        beta = dC_bgc/dCo2

        # gamma
        dC_cou = cLand_cou_data[155] - cLand_cou_data[0]
        dT_cou = tas_cou_data[155] - tas_cou_data[0]
        dC_res = dC_cou - dC_bgc
        gamma = dC_res/dT_cou

        print('T*=0', model, beta, gamma)

        # saving
        histbgc_beta[model_i] = beta
        histbgc_gamma[model_i] = gamma
        np.save('saved_data/histbgc_beta.npy', histbgc_beta.data)
        np.save('saved_data/histbgc_gamma.npy', histbgc_gamma.data)

        np.save('saved_data/cmip6_'+model+'_T_COU.npy', tas_cou_data.data)
        np.save('saved_data/cmip6_'+model+'_co2.npy', co2_array.data)
        np.save('saved_data/cmip6_'+model+'_cLand_COU.npy', cLand_cou_data.data)
        np.save('saved_data/cmip6_'+model+'_cLand_BGC.npy', cLand_bgc_data.data)
        np.save('saved_data/cmip6_timedimension.npy', time_dimension.data)



ACCESS-ESM1-5


/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")


T*=0 ACCESS-ESM1-5 197.1083069417026 -40.260094876013625
CanESM5


/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")


T*=0 CanESM5 3.6655509767476637 -14.418239545407515
CNRM-ESM2-1


...

T*=0 MIROC-ES2L 111.51597027296799 -68.96294618290268
NorESM2-LM


/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")


T*=0 NorESM2-LM -45.38416563445902 -17.59220948038796
UKESM1-0-LL


/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")
/Users/rebeccamayvarney/anaconda3/envs/rmv/lib/python3.11/site-packages/iris/analysis/cartography.py:413: UserWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn("Using DEFAULT_SPHERICAL_EARTH_RADIUS.")


T*=0 UKESM1-0-LL -16.2654315842912 -15.390488841751383
